# 01 — Raw audit, deterministic cleaning and coverage
Run on Dataproc Python 3. All exclusions are explicit; original CSVs remain unchanged.

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys

from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path[:0] = [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]

from rainfall.config import load_config
from rainfall.runtime import spark_action

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")


def cloud_read(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Cloud command failed.")
    return result.stdout

In [ ]:
from scripts.run_quality_audit import read_manifest, selected_files, run_audit, make_input_frame
from google.cloud import storage

manifest_path = PROJECT_ROOT / "manifests" / "input_manifest.json"
cloud_read(["storage", "cp", f"gs://{BUCKET}/manifests/input_manifest.json", str(manifest_path)])
manifest, manifest_sha256 = read_manifest(manifest_path)
files = selected_files(manifest)
client = storage.Client()
hash_results = []
for record in files:
    blob = client.bucket(BUCKET).blob(record["relative_path"])
    digest = hashlib.sha256()
    with blob.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    hash_results.append({"year": record["year"], "sha256_matches": digest.hexdigest() == record["sha256"]})
display(hash_results)
if not all(r["sha256_matches"] for r in hash_results):
    raise RuntimeError("At least one cloud input differs from the frozen local manifest.")

In [ ]:
with spark_action(spark, RUN_ID, "raw_quality_audit"):
    run_audit(spark, files, manifest_sha256, f"gs://{BUCKET}", f"{ROOT}/audit")
    spark.read.json(f"{ROOT}/audit/quality_by_year").orderBy("year").show(8, truncate=False)
    spark.read.json(f"{ROOT}/audit/reading_type_unit_distribution").show(30, truncate=False)

In [ ]:
from rainfall.cleaning import clean_observations, coverage
with spark_action(spark, RUN_ID, "deterministic_cleaning"):
    raw = make_input_frame(spark, files, f"gs://{BUCKET}")
    clean, rejected, quality, exclusions, alignment = clean_observations(raw, config)
    clean = clean.persist(StorageLevel.DISK_ONLY)
    clean.write.mode("errorifexists").partitionBy("year").parquet(f"{ROOT}/cleaned")
    rejected.write.mode("errorifexists").partitionBy("_source_year").parquet(f"{ROOT}/rejected")
    quality.write.mode("errorifexists").json(f"{ROOT}/quality_by_year")
    exclusions.write.mode("errorifexists").json(f"{ROOT}/exclusion_counts")
    alignment.write.mode("errorifexists").json(f"{ROOT}/alignment_distribution")
    quality.orderBy("year").show(8, truncate=False)
    exclusions.orderBy("_source_year", "exclusion_reason").show(100, truncate=False)
    raw.unpersist()

In [ ]:
with spark_action(spark, RUN_ID, "coverage_and_gaps"):
    clean = spark.read.parquet(f"{ROOT}/cleaned")
    months, years, days, gaps = coverage(clean)
    for name, frame in [("coverage_month", months), ("coverage_year", years), ("coverage_day", days), ("gaps", gaps)]:
        frame.write.mode("errorifexists").parquet(f"{ROOT}/{name}")
        frame.orderBy(*frame.columns[:2]).show(20, truncate=False)
    expected = sum(r["row_count"] for r in manifest["files"])
    excluded = spark.read.json(f"{ROOT}/exclusion_counts").agg(F.sum("count")).first()[0] or 0
    retained = clean.count()
    assert retained + excluded == expected, "Disjoint attrition counts must reconcile to raw rows."
    assert clean.groupBy("station_id", "epoch").count().where("count > 1").limit(1).count() == 0
    display({"raw_rows": expected, "retained_rows": retained, "excluded_rows": excluded})

In [ ]:
spark.stop()
print("Audit and cleaning complete. Review and capture every action's Spark UI evidence.")